In [ ]:

import os


os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import random
import multiprocessing
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import csv


PROFILE = []

def tick(name, t0):
    PROFILE.append((name, time.perf_counter() - t0))

def now():
    return time.perf_counter()

PROFILE_CSV = "stage_profile.csv"
EPOCH_PROFILE_CSV = "epoch_profile.csv"
BATCH_PROFILE_CSV = "batch_profile.csv"


t0 = now()

SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

NUM_CPU_CORES = multiprocessing.cpu_count()


tf.config.threading.set_intra_op_parallelism_threads(NUM_CPU_CORES)
tf.config.threading.set_inter_op_parallelism_threads(NUM_CPU_CORES)

tick("environment_setup", t0)

print("TensorFlow version :", tf.__version__)
print("Execution device   : CPU only")
print("CPU cores used     :", NUM_CPU_CORES)


t0 = now()

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test  = y_test.squeeze()

tick("dataset_load_and_preprocess", t0)


t0 = now()

VALIDATION_SPLIT = 0.1
n = x_train.shape[0]
idx = np.random.permutation(n)
val_n = int(n * VALIDATION_SPLIT)

x_val, y_val = x_train[idx[:val_n]], y_train[idx[:val_n]]
x_train, y_train = x_train[idx[val_n:]], y_train[idx[val_n:]]

tick("train_val_split", t0)


t0 = now()

BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000, seed=SEED)
    .map(lambda x, y: (x, y), num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(lambda x, y: (x, y), num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


for _ in train_ds.take(5):
    pass

tick("tfdata_pipeline_build", t0)

print("Dataset pipeline ready")


def resnet_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same",
                      use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 use_bias=False,
                                 kernel_regularizer=keras.regularizers.l2(1e-4))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.Activation("relu")(x)


def build_resnet18_cifar():
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(64, 3, padding="same",
                      use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = resnet_block(x, 64)
    x = resnet_block(x, 64)

    x = resnet_block(x, 128, stride=2)
    x = resnet_block(x, 128)

    x = resnet_block(x, 256, stride=2)
    x = resnet_block(x, 256)

    x = resnet_block(x, 512, stride=2)
    x = resnet_block(x, 512)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return keras.Model(inputs, outputs)


t0 = now()

model = build_resnet18_cifar()

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.1,
    decay_steps=50000
)

optimizer = keras.optimizers.SGD(
    learning_rate=lr_schedule,
    momentum=0.9
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

tick("model_build_and_compile", t0)

model.summary()


class TimeHistory(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.epoch_times = []
        self.batch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(
            (epoch, time.perf_counter() - self.epoch_start)
        )

    def on_train_batch_begin(self, batch, logs=None):
        self.batch_start = time.perf_counter()

    def on_train_batch_end(self, batch, logs=None):
        self.batch_times.append(
            (batch, time.perf_counter() - self.batch_start)
        )

t0 = now()
model.train_on_batch(x_train[:128], y_train[:128])
tick("single_train_step", t0)


EPOCHS = 10
steps_per_epoch = x_train.shape[0] // BATCH_SIZE
validation_steps = x_val.shape[0] // BATCH_SIZE

time_callback = TimeHistory()

t0 = now()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[time_callback],
    verbose=1
)
tick("full_training_time", t0)

t0 = now()
model.evaluate(val_ds, verbose=0)
tick("validation_time", t0)

t0 = now()
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
tick("test_inference_time", t0)

print("Final Test Accuracy:", test_acc)


with open(PROFILE_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["stage", "time_sec"])
    for k, v in PROFILE:
        writer.writerow([k, v])

with open(EPOCH_PROFILE_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "time_sec"])
    for e, t in time_callback.epoch_times:
        writer.writerow([e, t])

with open(BATCH_PROFILE_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["batch", "time_sec"])
    for b, t in time_callback.batch_times:
        writer.writerow([b, t])


print("\nPROFILING SUMMARY =====================")
for k, v in PROFILE:
    print(f"{k:30s}: {v:.4f} sec")

print("\nProfiling CSV files generated:")
print(" -", PROFILE_CSV)
print(" -", EPOCH_PROFILE_CSV)
print(" -", BATCH_PROFILE_CSV)

TensorFlow version : 2.19.0
Execution device   : CPU only
CPU cores used     : 4
Dataset pipeline ready


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_40 (Conv2D)  │ (None, 32, 32,    │      1,728 │ input_layer_2[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_40[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_34       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_41 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_34[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_41[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_35       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_35[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_42[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation_34[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_36       │ (None, 32, 32,    │          0 │ add_16[0][0]      │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_43 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_36[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_43[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_37       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_44 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_37[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_44[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 11,183,562 (42.66 MB)

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1799s 5s/step - loss: 3.3104 - sparse_categorical_accuracy: 0.2377 - val_loss: 2.6041 - val_sparse_categorical_accuracy: 0.2935
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1822s 5s/step - loss: 1.9601 - sparse_categorical_accuracy: 0.4493 - val_loss: 2.0470 - val_sparse_categorical_accuracy: 0.4237
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1833s 5s/step - loss: 1.6379 - sparse_categorical_accuracy: 0.5655 - val_loss: 1.6902 - val_sparse_categorical_accuracy: 0.5601
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1882s 5s/step - loss: 1.3470 - sparse_categorical_accuracy: 0.6581 - val_loss: 1.7384 - val_sparse_categorical_accuracy: 0.5773
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1800s 5s/step - loss: 1.1346 - sparse_categorical_accuracy: 0.7309 - val_loss: 1.1796 - val_sparse_categorical_accuracy: 0.7125
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 1825s 5s/step - loss: 0.9864 - sparse_categorical_accuracy: 0.7812 - val_loss: 1.2410 - val_sparse_categorical_accuracy: